# Hindi Coreference Resolution with AI4Bharat Embeddings

This notebook builds a compact end-to-end baseline for Hindi coreference resolution. It uses AI4Bharat IndicBERT v2, `ai4bharat/IndicBERTv2-MLM-only`, to create contextual word and mention embeddings, then trains a mention-pair classifier and turns pair scores into entity clusters.

References used for the model choice:
- AI4Bharat IndicBERT v2 collection: https://huggingface.co/collections/ai4bharat/indicbert-v2
- Model card for `ai4bharat/IndicBERTv2-MLM-only`: https://huggingface.co/ai4bharat/IndicBERTv2-MLM-only
- AI4Bharat IndicBERT page: https://indicnlp.ai4bharat.org/pages/indic-bert/

The sample dataset below is deliberately small so the whole notebook can run on a laptop. For a production system, replace it with a real annotated Hindi coreference corpus and keep the same functions.

## 1. Install dependencies

Run this once in a fresh Colab, Kaggle, or local Jupyter environment. The model download requires internet access the first time.

## 2. Imports and configuration

In [1]:
%pip install -q torch transformers scikit-learn pandas numpy tqdm matplotlib

In [2]:
from __future__ import annotations

import itertools
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

RANDOM_STATE = 7
MODEL_NAME = 'ai4bharat/IndicBERTv2-MLM-only'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f'Using device: {DEVICE}')

Using device: cuda


## 3. A tiny Hindi coreference dataset

Coreference needs annotated mentions and cluster ids. The toy data below includes names, nominals, pronouns, and singletons. Singletons are useful because they create negative training pairs.

In [3]:
@dataclass
class Mention:
    mention_id: str
    doc_id: str
    text: str
    start: int
    end: int
    cluster_id: str
    mention_type: str
    position: int = 0


@dataclass
class Document:
    doc_id: str
    text: str
    mentions: list[Mention]


def find_nth(text: str, needle: str, occurrence: int = 0) -> tuple[int, int]:
    offset = 0
    start = -1
    for _ in range(occurrence + 1):
        start = text.find(needle, offset)
        if start < 0:
            raise ValueError(f'Could not find occurrence {occurrence} of {needle!r} in {text!r}')
        offset = start + len(needle)
    return start, start + len(needle)


def make_doc(doc_id: str, text: str, specs: list[tuple[str, int, str, str]]) -> Document:
    mentions: list[Mention] = []
    for idx, (surface, occurrence, cluster_id, mention_type) in enumerate(specs):
        start, end = find_nth(text, surface, occurrence)
        mentions.append(
            Mention(
                mention_id=f'{doc_id}-m{idx}',
                doc_id=doc_id,
                text=surface,
                start=start,
                end=end,
                cluster_id=cluster_id,
                mention_type=mention_type,
            )
        )
    mentions = sorted(mentions, key=lambda m: (m.start, m.end))
    for position, mention in enumerate(mentions):
        mention.position = position
    return Document(doc_id=doc_id, text=text, mentions=mentions)


documents = [
    make_doc(
        'd1',
        'सीमा दिल्ली गई। वह सम्मेलन में बोली। सीमा ने कहा कि उसका शोध तैयार है।',
        [
            ('सीमा', 0, 'd1-person', 'name'),
            ('दिल्ली', 0, 'd1-place', 'name'),
            ('वह', 0, 'd1-person', 'pronoun'),
            ('सम्मेलन', 0, 'd1-event', 'nominal'),
            ('सीमा', 1, 'd1-person', 'name'),
            ('उसका', 0, 'd1-person', 'pronoun'),
            ('शोध', 0, 'd1-research', 'nominal'),
        ],
    ),
    make_doc(
        'd2',
        'राहुल ने नई कार खरीदी। गाड़ी लाल है। उसने उसे अपने भाई को दिखाया।',
        [
            ('राहुल', 0, 'd2-person', 'name'),
            ('नई कार', 0, 'd2-car', 'nominal'),
            ('गाड़ी', 0, 'd2-car', 'nominal'),
            ('उसने', 0, 'd2-person', 'pronoun'),
            ('उसे', 0, 'd2-car', 'pronoun'),
            ('अपने', 0, 'd2-person', 'pronoun'),
            ('भाई', 0, 'd2-brother', 'nominal'),
        ],
    ),
    make_doc(
        'd3',
        'प्रधानमंत्री ने योजना की घोषणा की। उन्होंने कहा कि यह योजना किसानों की मदद करेगी।',
        [
            ('प्रधानमंत्री', 0, 'd3-person', 'nominal'),
            ('योजना', 0, 'd3-plan', 'nominal'),
            ('उन्होंने', 0, 'd3-person', 'pronoun'),
            ('यह योजना', 0, 'd3-plan', 'nominal'),
            ('किसानों', 0, 'd3-farmers', 'nominal'),
        ],
    ),
    make_doc(
        'd4',
        'माया ने कविता को पत्र भेजा। उसने उसे जवाब दिया। कविता ने कहा कि वह खुश है।',
        [
            ('माया', 0, 'd4-maya', 'name'),
            ('कविता', 0, 'd4-kavita', 'name'),
            ('पत्र', 0, 'd4-letter', 'nominal'),
            ('उसने', 0, 'd4-kavita', 'pronoun'),
            ('उसे', 0, 'd4-maya', 'pronoun'),
            ('जवाब', 0, 'd4-answer', 'nominal'),
            ('कविता', 1, 'd4-kavita', 'name'),
            ('वह', 0, 'd4-kavita', 'pronoun'),
        ],
    ),
    make_doc(
        'd5',
        'कंपनी ने नया फोन लॉन्च किया। उसने बताया कि यह फोन भारत में बनेगा। ग्राहक इसे पसंद कर रहे हैं।',
        [
            ('कंपनी', 0, 'd5-company', 'nominal'),
            ('नया फोन', 0, 'd5-phone', 'nominal'),
            ('उसने', 0, 'd5-company', 'pronoun'),
            ('यह फोन', 0, 'd5-phone', 'nominal'),
            ('भारत', 0, 'd5-place', 'name'),
            ('ग्राहक', 0, 'd5-customers', 'nominal'),
            ('इसे', 0, 'd5-phone', 'pronoun'),
        ],
    ),
    make_doc(
        'd6',
        'मोहन और सोहन स्कूल गए। मोहन ने कहा कि वह परीक्षा की तैयारी करेगा। सोहन ने कहा कि उसे भी पढ़ना है।',
        [
            ('मोहन', 0, 'd6-mohan', 'name'),
            ('सोहन', 0, 'd6-sohan', 'name'),
            ('स्कूल', 0, 'd6-school', 'nominal'),
            ('मोहन', 1, 'd6-mohan', 'name'),
            ('वह', 0, 'd6-mohan', 'pronoun'),
            ('परीक्षा', 0, 'd6-exam', 'nominal'),
            ('सोहन', 1, 'd6-sohan', 'name'),
            ('उसे', 0, 'd6-sohan', 'pronoun'),
        ],
    ),
]

rows = []
for doc in documents:
    for mention in doc.mentions:
        rows.append(
            {
                'doc_id': doc.doc_id,
                'mention_id': mention.mention_id,
                'text': mention.text,
                'span': (mention.start, mention.end),
                'cluster_id': mention.cluster_id,
                'type': mention.mention_type,
                'context': doc.text[max(0, mention.start - 16): mention.end + 16],
            }
        )

mentions_df = pd.DataFrame(rows)
mentions_df

,doc_id,mention_id,text,span,cluster_id,type,context
0,d1,d1-m0,सीमा,"(0, 4)",d1-person,name,सीमा दिल्ली गई। वह स
1,d1,d1-m1,दिल्ली,"(5, 11)",d1-place,name,सीमा दिल्ली गई। वह सम्मेलन
2,d1,d1-m2,वह,"(16, 18)",d1-person,pronoun,सीमा दिल्ली गई। वह सम्मेलन में बोल
3,d1,d1-m3,सम्मेलन,"(19, 26)",d1-event,nominal,ा दिल्ली गई। वह सम्मेलन में बोली। सीमा
4,d1,d1-m4,सीमा,"(37, 41)",d1-person,name,्मेलन में बोली। सीमा ने कहा कि उसका
5,d1,d1-m5,उसका,"(52, 56)",d1-person,pronoun,सीमा ने कहा कि उसका शोध तैयार है।
6,d1,d1-m6,शोध,"(57, 60)",d1-research,nominal,ने कहा कि उसका शोध तैयार है।
7,d2,d2-m0,राहुल,"(0, 5)",d2-person,name,राहुल ने नई कार खरीदी
8,d2,d2-m1,नई कार,"(9, 15)",d2-car,nominal,राहुल ने नई कार खरीदी। गाड़ी ला
9,d2,d2-m2,गाड़ी,"(23, 28)",d2-car,nominal,े नई कार खरीदी। गाड़ी लाल है। उसने उस


## 3.1. Load the generated CSV dataset

The file `hindi_coref_dataset.csv` contains a larger synthetic dataset with one row per mention. If the CSV is present beside this notebook, the next cell replaces the tiny sample above with that larger dataset. In Google Colab, the cell will ask you to upload the CSV if it is not already available in the runtime.

In [4]:
CSV_DATASET_PATH = 'hindi_coref_dataset.csv'


def upload_csv_in_colab_if_needed(path: str) -> None:
    if Path(path).exists():
        return
    try:
        from google.colab import files
    except ModuleNotFoundError:
        return

    print(f'Upload {path} to train on the larger dataset.')
    uploaded = files.upload()
    if path not in uploaded and not Path(path).exists():
        uploaded_names = ', '.join(uploaded.keys()) or 'no files'
        raise FileNotFoundError(f'Expected {path}, but uploaded: {uploaded_names}')


def load_coref_csv(path: str) -> list[Document]:
    df = pd.read_csv(path)
    required_columns = {
        'doc_id',
        'doc_text',
        'mention_id',
        'mention_order',
        'mention_text',
        'mention_start',
        'mention_end',
        'cluster_id',
        'mention_type',
    }
    missing = sorted(required_columns - set(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    loaded_documents: list[Document] = []
    df = df.sort_values(['doc_id', 'mention_order']).reset_index(drop=True)

    for doc_id, group in df.groupby('doc_id', sort=False):
        doc_texts = group['doc_text'].unique()
        if len(doc_texts) != 1:
            raise ValueError(f'Document {doc_id} has multiple doc_text values.')
        text = str(doc_texts[0])
        mentions: list[Mention] = []

        for _, row in group.iterrows():
            start = int(row['mention_start'])
            end = int(row['mention_end'])
            surface = str(row['mention_text'])
            actual = text[start:end]
            if actual != surface:
                raise ValueError(
                    f'Span mismatch in {doc_id}: expected {surface!r}, found {actual!r} at {(start, end)}'
                )
            mentions.append(
                Mention(
                    mention_id=str(row['mention_id']),
                    doc_id=str(row['doc_id']),
                    text=surface,
                    start=start,
                    end=end,
                    cluster_id=str(row['cluster_id']),
                    mention_type=str(row['mention_type']),
                    position=int(row['mention_order']),
                )
            )
        loaded_documents.append(Document(doc_id=str(doc_id), text=text, mentions=mentions))

    return loaded_documents


try:
    upload_csv_in_colab_if_needed(CSV_DATASET_PATH)
    documents = load_coref_csv(CSV_DATASET_PATH)
    rows = []
    for doc in documents:
        for mention in doc.mentions:
            rows.append(
                {
                    'doc_id': doc.doc_id,
                    'mention_id': mention.mention_id,
                    'text': mention.text,
                    'span': (mention.start, mention.end),
                    'cluster_id': mention.cluster_id,
                    'type': mention.mention_type,
                    'context': doc.text[max(0, mention.start - 16): mention.end + 16],
                }
            )
    mentions_df = pd.DataFrame(rows)
    print(f'Loaded {len(documents)} documents and {len(mentions_df)} mentions from {CSV_DATASET_PATH}.')
except FileNotFoundError:
    print(f'{CSV_DATASET_PATH} not found; using the tiny in-notebook sample dataset.')

mentions_df.head(12)

Loaded 76 documents and 548 mentions from hindi_coref_dataset.csv.


,doc_id,mention_id,text,span,cluster_id,type,context
0,hc001,hc001-m00,सीमा,"(0, 4)",hc001-person,name,सीमा दिल्ली गई। वह स
1,hc001,hc001-m01,दिल्ली,"(5, 11)",hc001-place,name,सीमा दिल्ली गई। वह सम्मेलन
2,hc001,hc001-m02,वह,"(16, 18)",hc001-person,pronoun,सीमा दिल्ली गई। वह सम्मेलन में बोल
3,hc001,hc001-m03,सम्मेलन,"(19, 26)",hc001-event,nominal,ा दिल्ली गई। वह सम्मेलन में बोली। सीमा
4,hc001,hc001-m04,सीमा,"(37, 41)",hc001-person,name,्मेलन में बोली। सीमा ने कहा कि उसका
5,hc001,hc001-m05,उसका,"(52, 56)",hc001-person,pronoun,सीमा ने कहा कि उसका शोध तैयार है।
6,hc001,hc001-m06,शोध,"(57, 60)",hc001-work,nominal,ने कहा कि उसका शोध तैयार है।
7,hc002,hc002-m00,रीना,"(0, 4)",hc002-person,name,रीना जयपुर गई। वह का
8,hc002,hc002-m01,जयपुर,"(5, 10)",hc002-place,name,रीना जयपुर गई। वह कार्यशाल
9,hc002,hc002-m02,वह,"(15, 17)",hc002-person,pronoun,रीना जयपुर गई। वह कार्यशाला में ब


## 4. Load AI4Bharat IndicBERT v2

The tokenizer offset map lets us align Hindi character spans with subword tokens. A mention vector is the mean of the contextual token vectors that overlap the mention span.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if not tokenizer.is_fast:
    raise RuntimeError('This notebook needs a fast tokenizer for character offset alignment.')

model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

hidden_size = getattr(model.config, 'hidden_size', 'unknown')
print(f'Loaded {MODEL_NAME} on {DEVICE}. Hidden size: {hidden_size}')

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loaded ai4bharat/IndicBERTv2-MLM-only on cuda. Hidden size: 768


In [6]:
@torch.inference_mode()
def embed_document(doc: Document, max_length: int = 256) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    encoded = tokenizer(
        doc.text,
        return_offsets_mapping=True,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )
    offsets = encoded.pop('offset_mapping')[0].tolist()
    input_ids = encoded['input_ids'][0]
    model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}
    output = model(**model_inputs)
    hidden = output.last_hidden_state[0].detach().cpu()

    token_rows = []
    for token_id, (start, end) in enumerate(offsets):
        if start == end:
            continue
        token_rows.append(
            {
                'token_id': token_id,
                'token': tokenizer.convert_ids_to_tokens(int(input_ids[token_id])),
                'start': start,
                'end': end,
                'text': doc.text[start:end],
            }
        )

    mention_vectors: dict[str, np.ndarray] = {}
    for mention in doc.mentions:
        token_ids = [
            token_id
            for token_id, (start, end) in enumerate(offsets)
            if start != end and end > mention.start and start < mention.end
        ]
        if not token_ids:
            raise ValueError(f'Mention {mention.text!r} in {doc.doc_id} was not aligned to any token.')
        vector = hidden[token_ids].mean(dim=0).numpy()
        vector = vector / (np.linalg.norm(vector) + 1e-12)
        mention_vectors[mention.mention_id] = vector

    return mention_vectors, pd.DataFrame(token_rows)


embeddings_by_doc: dict[str, dict[str, np.ndarray]] = {}
token_tables: dict[str, pd.DataFrame] = {}

for doc in tqdm(documents, desc='Embedding Hindi documents'):
    embeddings_by_doc[doc.doc_id], token_tables[doc.doc_id] = embed_document(doc)

first_doc_id = documents[0].doc_id
token_tables[first_doc_id].head(20)

Embedding Hindi documents:   0%|          | 0/76 [00:00<?, ?it/s]

,token_id,token,start,end,text
0,1,सीमा,0,4,सीमा
1,2,दिल्ली,5,11,दिल्ली
2,3,गई,12,14,गई
3,4,।,14,15,।
4,5,वह,16,18,वह
5,6,सम्मेलन,19,26,सम्मेलन
6,7,में,27,30,में
7,8,बोली,31,35,बोली
8,9,।,35,36,।
9,10,सीमा,37,41,सीमा


## 5. Inspect mention embedding similarity

This is a quick sanity check. Similarity alone is not coreference, but it is a useful feature for a resolver.

In [7]:
def cosine(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    return float(np.dot(vec_a, vec_b) / ((np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-12))


doc = documents[0]
sim_rows = []
for left, right in itertools.combinations(doc.mentions, 2):
    sim_rows.append(
        {
            'left': left.text,
            'right': right.text,
            'same_gold_cluster': left.cluster_id == right.cluster_id,
            'cosine': cosine(embeddings_by_doc[doc.doc_id][left.mention_id], embeddings_by_doc[doc.doc_id][right.mention_id]),
        }
    )

pd.DataFrame(sim_rows).sort_values('cosine', ascending=False).head(12)

,left,right,same_gold_cluster,cosine
3,सीमा,सीमा,True,0.896319
17,सम्मेलन,शोध,False,0.822759
2,सीमा,सम्मेलन,False,0.740041
0,सीमा,दिल्ली,False,0.722582
5,सीमा,शोध,False,0.677004
15,सम्मेलन,सीमा,False,0.674405
20,उसका,शोध,False,0.669722
7,दिल्ली,सम्मेलन,False,0.666930
8,दिल्ली,सीमा,False,0.657339
4,सीमा,उसका,True,0.623771


## 6. Build mention-pair training examples

For every mention, candidate antecedents are all earlier mentions in the same document. The label is `1` when both mentions belong to the same gold cluster.

In [8]:
HINDI_PRONOUNS = {
    'मैं', 'हम', 'तू', 'तुम', 'आप', 'वह', 'वे', 'यह', 'ये', 'उसने', 'उन्होंने',
    'उसे', 'उन्हें', 'इसने', 'इसे', 'उसका', 'उसकी', 'उसके', 'उनका', 'उनकी',
    'अपने', 'अपनी', 'खुद', 'स्वयं',
}

OBJECT_LIKE_PRONOUNS = {'उसे', 'इसे', 'उन्हें', 'इन्हें', 'उनको', 'इनको'}
PLURAL_OBJECT_LIKE_PRONOUNS = {'उन्हें', 'इन्हें', 'उनको', 'इनको'}
SUBJECT_LIKE_PRONOUNS = {'उसने', 'इसने', 'उन्होंने', 'इन्होंने'}
CLAUSE_BOUNDARY_RE = re.compile(r'[।!?]')


def is_pronoun_mention(mention: Mention) -> bool:
    return mention.text in HINDI_PRONOUNS or mention.mention_type == 'pronoun'


def is_current_clause_ne_subject(doc: Document, antecedent: Mention, anaphor: Mention) -> bool:
    between = doc.text[antecedent.end:anaphor.start]
    if CLAUSE_BOUNDARY_RE.search(between):
        return False
    return bool(re.fullmatch(r'\s*ने\s*', between))


def suppress_candidate_for_anaphor(doc: Document, antecedent: Mention, anaphor: Mention) -> bool:
    if anaphor.text in OBJECT_LIKE_PRONOUNS and is_pronoun_mention(antecedent):
        return True
    if anaphor.text in OBJECT_LIKE_PRONOUNS and is_current_clause_ne_subject(doc, antecedent, anaphor):
        return True
    return False


def normalize_surface(text: str) -> str:
    return ''.join(text.split())


def pair_feature_dict(doc: Document, antecedent: Mention, anaphor: Mention, doc_embeddings: dict[str, np.ndarray]) -> dict[str, float]:
    ant_vec = doc_embeddings[antecedent.mention_id]
    ana_vec = doc_embeddings[anaphor.mention_id]
    ant_norm = normalize_surface(antecedent.text)
    ana_norm = normalize_surface(anaphor.text)
    return {
        'cosine': cosine(ant_vec, ana_vec),
        'mention_gap': float(anaphor.position - antecedent.position),
        'char_gap': float(max(0, anaphor.start - antecedent.end)),
        'same_surface': float(ant_norm == ana_norm),
        'surface_contains': float(ant_norm in ana_norm or ana_norm in ant_norm),
        'antecedent_is_pronoun': float(is_pronoun_mention(antecedent)),
        'anaphor_is_pronoun': float(is_pronoun_mention(anaphor)),
        'object_like_pronoun_anaphor': float(anaphor.text in OBJECT_LIKE_PRONOUNS),
        'pronoun_to_pronoun_pair': float(is_pronoun_mention(antecedent) and is_pronoun_mention(anaphor)),
        'object_pronoun_to_pronoun': float(anaphor.text in OBJECT_LIKE_PRONOUNS and is_pronoun_mention(antecedent)),
        'antecedent_is_current_ne_subject': float(is_current_clause_ne_subject(doc, antecedent, anaphor)),
        'antecedent_len': float(len(antecedent.text)),
        'anaphor_len': float(len(anaphor.text)),
        'same_first_char': float(bool(antecedent.text and anaphor.text and antecedent.text[0] == anaphor.text[0])),
    }


FEATURE_COLUMNS = [
    'cosine',
    'mention_gap',
    'char_gap',
    'same_surface',
    'surface_contains',
    'antecedent_is_pronoun',
    'anaphor_is_pronoun',
    'object_like_pronoun_anaphor',
    'pronoun_to_pronoun_pair',
    'object_pronoun_to_pronoun',
    'antecedent_is_current_ne_subject',
    'antecedent_len',
    'anaphor_len',
    'same_first_char',
]


def build_pair_table(docs: Iterable[Document], embeddings: dict[str, dict[str, np.ndarray]]) -> pd.DataFrame:
    rows = []
    for doc in docs:
        doc_embeddings = embeddings[doc.doc_id]
        for ana_index in range(1, len(doc.mentions)):
            anaphor = doc.mentions[ana_index]
            for ant_index in range(ana_index):
                antecedent = doc.mentions[ant_index]
                features = pair_feature_dict(doc, antecedent, anaphor, doc_embeddings)
                rows.append(
                    {
                        'doc_id': doc.doc_id,
                        'antecedent_id': antecedent.mention_id,
                        'anaphor_id': anaphor.mention_id,
                        'antecedent': antecedent.text,
                        'anaphor': anaphor.text,
                        'label': int(antecedent.cluster_id == anaphor.cluster_id),
                        **features,
                    }
                )
    return pd.DataFrame(rows)


pair_df = build_pair_table(documents, embeddings_by_doc)
pair_df[['doc_id', 'antecedent', 'anaphor', 'label', 'cosine', 'mention_gap', 'same_surface']].head(12)

,doc_id,antecedent,anaphor,label,cosine,mention_gap,same_surface
0,hc001,सीमा,दिल्ली,0,0.722582,1.0,0.0
1,hc001,सीमा,वह,1,0.153673,2.0,0.0
2,hc001,दिल्ली,वह,0,0.223621,1.0,0.0
3,hc001,सीमा,सम्मेलन,0,0.740041,3.0,0.0
4,hc001,दिल्ली,सम्मेलन,0,0.666930,2.0,0.0
5,hc001,वह,सम्मेलन,0,0.050012,1.0,0.0
6,hc001,सीमा,सीमा,1,0.896319,4.0,1.0
7,hc001,दिल्ली,सीमा,0,0.657339,3.0,0.0
8,hc001,वह,सीमा,1,0.149533,2.0,0.0
9,hc001,सम्मेलन,सीमा,0,0.674405,1.0,0.0


## 7. Train a pairwise coreference scorer

This baseline keeps the encoder frozen and trains only a lightweight logistic regression classifier. With more data, you can replace this with a neural pair scorer and optionally fine-tune IndicBERT.

In [9]:
if len(documents) < 2:
    raise ValueError('At least two documents are needed for a train/test split.')

shuffled_documents = documents.copy()
random.Random(RANDOM_STATE).shuffle(shuffled_documents)
split_index = max(1, min(len(shuffled_documents) - 1, int(len(shuffled_documents) * 0.8)))
train_docs = shuffled_documents[:split_index]
test_docs = shuffled_documents[split_index:]
train_ids = {doc.doc_id for doc in train_docs}
test_ids = {doc.doc_id for doc in test_docs}

train_pairs = pair_df[pair_df['doc_id'].isin(train_ids)].reset_index(drop=True)
test_pairs = pair_df[pair_df['doc_id'].isin(test_ids)].reset_index(drop=True)

pair_scorer = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
)
pair_scorer.fit(train_pairs[FEATURE_COLUMNS], train_pairs['label'])

test_prob = pair_scorer.predict_proba(test_pairs[FEATURE_COLUMNS])[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print(classification_report(test_pairs['label'], test_pred, zero_division=0))

inspect = test_pairs[['doc_id', 'antecedent', 'anaphor', 'label']].copy()
inspect['p_coref'] = test_prob.round(3)
inspect.sort_values('p_coref', ascending=False).head(15)

              precision    recall  f1-score   support

           0       0.93      0.82      0.87       244
           1       0.61      0.82      0.70        83

    accuracy                           0.82       327
   macro avg       0.77      0.82      0.78       327
weighted avg       0.85      0.82      0.83       327



,doc_id,antecedent,anaphor,label,p_coref
323,hc069,दवाइयां,दवाइयां,1,1.000
281,hc066,किताबें,किताबें,1,1.000
219,hc051,अमोल,अमोल,1,1.000
288,hc067,माया,माया,1,0.999
260,hc065,फल,फल,1,0.999
191,hc047,वंदना,वंदना,1,0.999
48,hc008,दीपा,दीपा,1,0.999
267,hc066,सीमा,सीमा,1,0.999
202,hc047,आरती,आरती,1,0.999
174,hc042,नेहा,नेहा,1,0.999


## 8. Convert pair scores into clusters

The resolver links each mention to the highest-scoring previous mention when the score crosses a threshold. Linked mentions are merged with union-find.

In [10]:
class UnionFind:
    def __init__(self, items: Iterable[str]):
        self.parent = {item: item for item in items}

    def find(self, item: str) -> str:
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: str, right: str) -> None:
        left_root = self.find(left)
        right_root = self.find(right)
        if left_root != right_root:
            self.parent[right_root] = left_root


def predict_clusters(
    doc: Document,
    doc_embeddings: dict[str, np.ndarray],
    scorer,
    threshold: float = 0.55,
) -> tuple[dict[str, list[Mention]], pd.DataFrame]:
    mention_ids = [mention.mention_id for mention in doc.mentions]
    uf = UnionFind(mention_ids)
    decisions = []

    for ana_index in range(1, len(doc.mentions)):
        anaphor = doc.mentions[ana_index]
        candidates = []
        feature_rows = []
        for ant_index in range(ana_index):
            antecedent = doc.mentions[ant_index]
            feature_rows.append(pair_feature_dict(doc, antecedent, anaphor, doc_embeddings))
            candidates.append(antecedent)

        candidate_features = pd.DataFrame(feature_rows)[FEATURE_COLUMNS]
        raw_probabilities = scorer.predict_proba(candidate_features)[:, 1]
        probabilities = raw_probabilities.copy()
        suppressed_candidates = []

        for candidate_index, candidate in enumerate(candidates):
            if suppress_candidate_for_anaphor(doc, candidate, anaphor):
                probabilities[candidate_index] = 0.0
                suppressed_candidates.append(candidate.text)

        if anaphor.text in PLURAL_OBJECT_LIKE_PRONOUNS:
            content_candidate_indices = [
                candidate_index
                for candidate_index, candidate in enumerate(candidates)
                if not suppress_candidate_for_anaphor(doc, candidate, anaphor) and not is_pronoun_mention(candidate)
            ]
            if content_candidate_indices:
                closest_content_index = content_candidate_indices[-1]
                probabilities[closest_content_index] = max(float(probabilities.max()) + 1e-3, threshold)

        best_index = int(np.argmax(probabilities))
        best_antecedent = candidates[best_index]
        best_probability = float(probabilities[best_index])
        raw_probability = float(raw_probabilities[best_index])
        linked = best_probability >= threshold

        if linked:
            uf.union(best_antecedent.mention_id, anaphor.mention_id)

        decisions.append(
            {
                'doc_id': doc.doc_id,
                'anaphor': anaphor.text,
                'best_antecedent': best_antecedent.text,
                'p_coref': round(best_probability, 3),
                'raw_p_coref': round(raw_probability, 3),
                'linked': linked,
                'suppressed_candidates': ' | '.join(suppressed_candidates),
                'gold_same_cluster': best_antecedent.cluster_id == anaphor.cluster_id,
            }
        )

    grouped: dict[str, list[Mention]] = defaultdict(list)
    for mention in doc.mentions:
        grouped[uf.find(mention.mention_id)].append(mention)
    return dict(grouped), pd.DataFrame(decisions)


def gold_clusters(doc: Document) -> dict[str, list[Mention]]:
    grouped: dict[str, list[Mention]] = defaultdict(list)
    for mention in doc.mentions:
        grouped[mention.cluster_id].append(mention)
    return dict(grouped)


def cluster_pair_set(clusters: dict[str, list[Mention]]) -> set[tuple[str, str]]:
    pairs: set[tuple[str, str]] = set()
    for members in clusters.values():
        ids = sorted(mention.mention_id for mention in members)
        pairs.update(itertools.combinations(ids, 2))
    return pairs


def pairwise_coref_scores(docs: Iterable[Document], predicted: dict[str, dict[str, list[Mention]]]) -> dict[str, float]:
    gold_pairs: set[tuple[str, str]] = set()
    predicted_pairs: set[tuple[str, str]] = set()
    for doc in docs:
        gold_pairs |= cluster_pair_set(gold_clusters(doc))
        predicted_pairs |= cluster_pair_set(predicted[doc.doc_id])

    true_positive = len(gold_pairs & predicted_pairs)
    precision = true_positive / len(predicted_pairs) if predicted_pairs else 0.0
    recall = true_positive / len(gold_pairs) if gold_pairs else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'gold_pairs': len(gold_pairs), 'predicted_pairs': len(predicted_pairs)}


predicted_test = {}
decision_tables = []
for doc in test_docs:
    predicted_test[doc.doc_id], decisions = predict_clusters(doc, embeddings_by_doc[doc.doc_id], pair_scorer, threshold=0.55)
    decision_tables.append(decisions)

pd.DataFrame([pairwise_coref_scores(test_docs, predicted_test)])

,precision,recall,f1,gold_pairs,predicted_pairs
0,0.6375,0.614458,0.625767,83,80


## 8.1. MUC-style evaluation

MUC is a link-based coreference metric. For recall, each gold cluster is split into the predicted clusters that cover it; fewer splits mean better recall. For precision, the same calculation is run in the opposite direction, splitting predicted clusters by gold clusters.

In [14]:
def mention_to_cluster_map(clusters: dict[str, list[Mention]]) -> dict[str, str]:
    mapping = {}
    for cluster_id, members in clusters.items():
        for mention in members:
            mapping[mention.mention_id] = str(cluster_id)
    return mapping


def muc_link_counts(
    source_clusters: dict[str, list[Mention]],
    partition_clusters: dict[str, list[Mention]],
) -> tuple[int, int]:
    partition_map = mention_to_cluster_map(partition_clusters)
    numerator = 0
    denominator = 0

    for members in source_clusters.values():
        mention_ids = [mention.mention_id for mention in members]
        if len(mention_ids) <= 1:
            continue
        partitions = {partition_map.get(mention_id, f'missing:{mention_id}') for mention_id in mention_ids}
        numerator += len(mention_ids) - len(partitions)
        denominator += len(mention_ids) - 1

    return numerator, denominator


def muc_coref_scores(docs: Iterable[Document], predicted: dict[str, dict[str, list[Mention]]]) -> dict[str, float]:
    recall_num = recall_den = 0
    precision_num = precision_den = 0

    for doc in docs:
        gold = gold_clusters(doc)
        pred = predicted[doc.doc_id]

        doc_recall_num, doc_recall_den = muc_link_counts(gold, pred)
        doc_precision_num, doc_precision_den = muc_link_counts(pred, gold)

        recall_num += doc_recall_num
        recall_den += doc_recall_den
        precision_num += doc_precision_num
        precision_den += doc_precision_den

    precision = precision_num / precision_den if precision_den else 0.0
    recall = recall_num / recall_den if recall_den else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        'muc_precision': precision,
        'muc_recall': recall,
        'muc_f1': f1,
        'muc_precision_links': precision_den,
        'muc_recall_links': recall_den,
    }


pd.DataFrame([muc_coref_scores(test_docs, predicted_test)])

,muc_precision,muc_recall,muc_f1,muc_precision_links,muc_recall_links
0,0.735849,0.696429,0.715596,53,56


In [11]:
def clusters_to_frame(doc: Document, clusters: dict[str, list[Mention]], label: str) -> pd.DataFrame:
    rows = []
    for cluster_index, members in enumerate(clusters.values(), start=1):
        rows.append(
            {
                'doc_id': doc.doc_id,
                'cluster_kind': label,
                'cluster': cluster_index,
                'mentions': ' | '.join(mention.text for mention in members),
                'spans': [(mention.start, mention.end) for mention in members],
            }
        )
    return pd.DataFrame(rows)


frames = []
for doc in test_docs:
    frames.append(clusters_to_frame(doc, gold_clusters(doc), 'gold'))
    frames.append(clusters_to_frame(doc, predicted_test[doc.doc_id], 'predicted'))

display(pd.concat(frames, ignore_index=True))
display(pd.concat(decision_tables, ignore_index=True).sort_values('p_coref', ascending=False))

,doc_id,cluster_kind,cluster,mentions,spans
0,hc067,gold,1,माया | उसने | माया,"[(0, 4), (15, 19), (34, 38)]"
1,hc067,gold,2,फाइलें | उन्हें | फाइलें,"[(20, 26), (42, 48), (62, 68)]"
2,hc067,gold,3,मेज,"[(49, 52)]"
3,hc067,predicted,1,माया | उसने | माया,"[(0, 4), (15, 19), (34, 38)]"
4,hc067,predicted,2,फाइलें | उन्हें | फाइलें,"[(20, 26), (42, 48), (62, 68)]"
...,...,...,...,...,...
106,hc042,gold,4,जवाब,"[(61, 65)]"
107,hc042,predicted,1,नेहा | नेहा,"[(0, 4), (53, 57)]"
108,hc042,predicted,2,पूजा | पूजा,"[(8, 12), (28, 32)]"
109,hc042,predicted,3,संदेश | उसे | उसने,"[(16, 21), (36, 39), (48, 52)]"


,doc_id,anaphor,best_antecedent,p_coref,raw_p_coref,linked,suppressed_candidates,gold_same_cluster
16,hc066,किताबें,किताबें,1.000,1.000,True,,True
76,hc051,अमोल,अमोल,1.000,1.000,True,,True
62,hc069,दवाइयां,दवाइयां,1.000,1.000,True,,True
2,hc067,माया,माया,0.999,0.999,True,,True
13,hc066,सीमा,सीमा,0.999,0.999,True,,True
...,...,...,...,...,...,...,...,...
45,hc047,वंदना,आरती,0.001,0.001,False,,False
74,hc051,अमोल,डॉ भारती,0.001,0.001,False,,False
17,hc012,नया स्कूटर,विकास,0.000,0.000,False,,False
52,hc013,नया बैग,अरुण,0.000,0.000,False,,False


## 9. Run the resolver on new Hindi text

Mention detection is its own task. To keep this notebook focused on coreference, the helper below accepts exact mention strings in reading order. For a larger system, replace this with a Hindi NER or span detector.

In [12]:
def doc_from_mention_texts(doc_id: str, text: str, mention_texts: list[str]) -> Document:
    seen = Counter()
    specs = []
    for index, surface in enumerate(mention_texts):
        occurrence = seen[surface]
        seen[surface] += 1
        mention_type = 'pronoun' if surface in HINDI_PRONOUNS else 'nominal'
        specs.append((surface, occurrence, f'{doc_id}-singleton-{index}', mention_type))
    return make_doc(doc_id, text, specs)


def resolve_text(text: str, mention_texts: list[str], threshold: float = 0.55) -> tuple[pd.DataFrame, pd.DataFrame]:
    doc = doc_from_mention_texts('new-doc', text, mention_texts)
    doc_embeddings, _ = embed_document(doc)
    clusters, decisions = predict_clusters(doc, doc_embeddings, pair_scorer, threshold=threshold)
    decisions = decisions.drop(columns=['gold_same_cluster'], errors='ignore')
    return clusters_to_frame(doc, clusters, 'predicted'), decisions


new_text = 'रीना बाजार गई। उसने फल खरीदे। रीना ने उन्हें घर में रखा।'
new_mentions = ['रीना', 'उसने', 'फल', 'रीना', 'उन्हें', 'घर']

predicted_clusters, link_decisions = resolve_text(new_text, new_mentions, threshold=0.55)
display(predicted_clusters)
display(link_decisions.sort_values('p_coref', ascending=False))

,doc_id,cluster_kind,cluster,mentions,spans
0,new-doc,predicted,1,रीना | उसने | रीना,"[(0, 4), (15, 19), (30, 34)]"
1,new-doc,predicted,2,फल | उन्हें,"[(20, 22), (38, 44)]"
2,new-doc,predicted,3,घर,"[(45, 47)]"


,doc_id,anaphor,best_antecedent,p_coref,raw_p_coref,linked,suppressed_candidates
2,new-doc,रीना,रीना,0.999,0.999,True,
0,new-doc,उसने,रीना,0.788,0.788,True,
3,new-doc,उन्हें,फल,0.772,0.771,True,उसने | रीना
4,new-doc,घर,उन्हें,0.487,0.487,False,
1,new-doc,फल,उसने,0.190,0.190,False,


## 10. How to extend this baseline

- Replace the toy documents with real Hindi coreference annotations. Keep `Document` and `Mention` objects, or write a converter from JSON/CSV/CoNLL.
- Improve mention detection with a Hindi NER, dependency parser, or a trained span proposal model.
- Fine-tune the pair scorer on more examples. A common upgrade is a small neural network over `[antecedent_embedding, anaphor_embedding, elementwise_product, distance_features]`.
- Tune the clustering threshold on a validation set instead of using `0.55`.
- Add stricter evaluation metrics such as MUC, B-cubed, and CEAF when comparing with published systems.

This notebook is meant to be a clear, runnable foundation: AI4Bharat embeddings for Hindi mentions, supervised pair scoring, and cluster construction.